In [1]:
import os, time, random, numpy as np
import scipy.io as sio
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from scipy.sparse import coo_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# -------------------- Data Loading & Graph Construction --------------------
def load_dataset(data_path, dataset_name, config):
    file_path = os.path.join(data_path, config['file'])
    data = sio.loadmat(file_path)[config['data_key']]
    if config['file'] == config['gt_file']:
        labels = sio.loadmat(file_path)[config['gt_key']]
    else:
        gt_path = os.path.join(data_path, config['gt_file'])
        labels = sio.loadmat(gt_path)[config['gt_key']]
    if len(labels.shape) == 3:
        labels = labels.squeeze()
    return data, labels

def applyPCA(X, numComponents):
    original_shape = X.shape
    X_reshaped = X.reshape(-1, X.shape[-1])
    pca = PCA(n_components=min(numComponents, X.shape[-1]), whiten=True)
    X_pca = pca.fit_transform(X_reshaped)
    return X_pca.reshape(original_shape[0], original_shape[1], -1)

def extract_nodes_with_coords(X_pca, y, windowSize=5):
    margin = windowSize // 2
    nodes, labels, coords = [], [], []
    for r in range(margin, X_pca.shape[0] - margin):
        for c in range(margin, X_pca.shape[1] - margin):
            if y[r, c] > 0:
                nodes.append(X_pca[r, c, :])
                labels.append(y[r, c] - 1)
                coords.append((r, c))
    return np.array(nodes), np.array(labels), np.array(coords)

def build_adjacency(coords, max_dist=1):
    n_nodes = len(coords)
    rows, cols = [], []
    for i in range(n_nodes):
        for j in range(i+1, n_nodes):
            dist = max(abs(coords[i][0]-coords[j][0]), abs(coords[i][1]-coords[j][1]))
            if dist <= max_dist:
                rows.append(i); cols.append(j)
                rows.append(j); cols.append(i)
    values = np.ones(len(rows), dtype=np.float32)
    adj = coo_matrix((values, (rows, cols)), shape=(n_nodes, n_nodes))
    adj = adj + coo_matrix((np.ones(n_nodes), (range(n_nodes), range(n_nodes))), shape=(n_nodes, n_nodes))
    degree = np.array(adj.sum(axis=1)).flatten()
    degree_inv_sqrt = np.power(degree, -0.5, where=degree>0)
    D_inv_sqrt = coo_matrix((degree_inv_sqrt, (range(n_nodes), range(n_nodes))), shape=(n_nodes, n_nodes))
    adj_norm = D_inv_sqrt @ adj @ D_inv_sqrt
    adj_norm_coo = adj_norm.tocoo()
    adj_indices = torch.tensor([adj_norm_coo.row, adj_norm_coo.col], dtype=torch.long)
    adj_values = torch.tensor(adj_norm_coo.data, dtype=torch.float32)
    adj_shape = torch.Size([n_nodes, n_nodes])
    return torch.sparse_coo_tensor(adj_indices, adj_values, adj_shape).coalesce()

# -------------------- Self‑Adaptive Clustering --------------------
class AdaptiveClustering(nn.Module):
    def __init__(self, weight_shape):
        super().__init__()
        self.out_channels = weight_shape[0]
        self.D = int(np.prod(weight_shape[1:]))
        self.keep_ratio = nn.Parameter(torch.tensor(0.5))
        self.tau = nn.Parameter(torch.tensor(1.0))

    @property
    def K(self):
        k = int(max(2, torch.sigmoid(self.keep_ratio).item() * self.out_channels))
        return min(k, self.out_channels)

    def get_centroids(self):
        if not hasattr(self, '_centroids_buffer') or self._centroids_buffer.shape[0] != self.K:
            self.register_buffer('_centroids_buffer', torch.randn(self.K, self.D) * 0.01)
        return self._centroids_buffer

    def forward(self, W):
        W_flat = W.view(W.shape[0], -1)
        O, D = W_flat.shape
        K_eff = self.K
        centroids = self.get_centroids()[:K_eff]
        tau_val = self.tau.abs() + 0.05
        dist = torch.cdist(W_flat, centroids)
        A = F.softmax(-dist / tau_val, dim=1)
        W_compressed = torch.mm(A, centroids)
        return W_compressed.view_as(W), A

# -------------------- Compressible GCN Layer --------------------
class CompressibleGCNLayer(nn.Module):
    def __init__(self, in_features, out_features, compress_layers, gate_init=0.0, layer_name=""):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.layer_name = layer_name
        self.compress_layers = compress_layers
        if layer_name in compress_layers:
            self.compression_module = AdaptiveClustering(self.linear.weight.shape)
            self.gate = nn.Parameter(torch.tensor(gate_init))
        else:
            self.compression_module = None
            self.gate = None

    def get_compressed_weight(self):
        if self.compression_module is None:
            return self.linear.weight
        gate = torch.sigmoid(self.gate)
        if gate < 0.01:
            return self.linear.weight
        compressed, _ = self.compression_module(self.linear.weight)
        return (1 - gate) * self.linear.weight + gate * compressed

    def forward(self, x, adj):
        w = self.get_compressed_weight()
        x = F.linear(x, w, self.linear.bias)
        x = torch.spmm(adj, x)
        return x

    def sparsity_loss(self, lambda_base=1.5):
        if self.compression_module is None:
            return torch.tensor(0.0, device=DEVICE)
        gate = torch.sigmoid(self.gate)
        keep = torch.sigmoid(self.compression_module.keep_ratio)
        return lambda_base * (1 - gate) + 0.8 * keep

    def get_learned_K(self):
        if self.compression_module is None:
            return None
        return self.compression_module.K

    def finalize_compression(self, gate_threshold=0.4):
        if self.compression_module is None or torch.sigmoid(self.gate).item() < gate_threshold:
            return False
        mod = self.compression_module
        K = mod.K
        centroids = mod.get_centroids()
        orig_weight = self.linear.weight.data
        dist = torch.cdist(orig_weight, centroids)
        assignments = dist.argmin(dim=1)
        reconstructed = centroids[assignments]
        new_linear = nn.Linear(self.linear.in_features, K)
        new_linear.weight.data = reconstructed[:K]
        new_linear.bias.data = self.linear.bias.data[:K] if self.linear.bias is not None else None
        self.linear = new_linear
        self.compression_module = None
        self.gate = None
        return True

# -------------------- Scaled GCN (only first layer compressible) --------------------
class SelfAdaptiveGCN(nn.Module):
    def __init__(self, n_classes, n_features, hidden_dims=[256, 128],
                 compress_layers=None, gate_init=0.0):
        super().__init__()
        self.compress_layers = compress_layers if compress_layers is not None else []
        self.layers = nn.ModuleList()
        prev_dim = n_features
        for i, hdim in enumerate(hidden_dims):
            layer = CompressibleGCNLayer(prev_dim, hdim, self.compress_layers, gate_init, f"gcn{i+1}")
            self.layers.append(layer)
            prev_dim = hdim
        # Final classifier (never compressed)
        self.classifier = nn.Linear(prev_dim, n_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x, adj):
        for layer in self.layers:
            x = layer(x, adj)
            x = F.relu(x)
            x = self.dropout(x)
        return self.classifier(x)

    def sparsity_loss(self):
        loss = torch.tensor(0.0, device=DEVICE)
        for layer in self.layers:
            loss += layer.sparsity_loss()
        return loss / len(self.layers)

    def get_learned_K(self):
        k_vals = {}
        for i, layer in enumerate(self.layers):
            k = layer.get_learned_K()
            if k is not None:
                k_vals[f"gcn{i+1}"] = k
        return k_vals

    def finalize_compression(self, gate_threshold=0.4):
        # Compress only the first layer; then update subsequent layers' input dimensions
        if len(self.layers) == 0:
            return
        compressed = self.layers[0].finalize_compression(gate_threshold)
        if compressed:
            new_hidden = self.layers[0].linear.out_features
            # Update the second layer (if exists) to match the new hidden size
            if len(self.layers) > 1:
                old_linear = self.layers[1].linear
                new_linear = nn.Linear(new_hidden, old_linear.out_features)
                with torch.no_grad():
                    new_linear.weight.data = old_linear.weight.data[:, :new_hidden]
                    new_linear.bias.data = old_linear.bias.data
                self.layers[1].linear = new_linear
                # Disable compression for the second layer (its input dimension has changed)
                self.layers[1].compression_module = None
                self.layers[1].gate = None
        # Do not try to compress the second layer separately

# -------------------- Training Functions --------------------
def train_gcn_two_stage(model, adj, features, labels, train_mask, val_mask,
                        epochs_stage1=150, epochs_stage2=60, final_lambda=1.8, lr=0.005):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    ce_loss = nn.CrossEntropyLoss()
    adj = adj.to(DEVICE)
    features = features.to(DEVICE)
    labels = labels.to(DEVICE)

    print("\n--- Stage 1: Training without compression loss ---")
    best_val_acc = 0
    best_state = None
    patience = 30
    no_improve = 0

    for epoch in range(epochs_stage1):
        model.train()
        optimizer.zero_grad()
        out = model(features, adj)
        loss = ce_loss(out[train_mask], labels[train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_val = model(features, adj)
            pred = out_val.argmax(dim=1)
            val_acc = (pred[val_mask] == labels[val_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience and best_val_acc > 0.95:
                print(f"Early stopping at epoch {epoch+1} (best val_acc={best_val_acc:.4f})")
                model.load_state_dict(best_state)
                break
        if (epoch+1) % 20 == 0:
            print(f"Stage1 Epoch {epoch+1}/{epochs_stage1} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}")

    print("\n--- Stage 2: Adding compression loss (K evolution) ---")
    for epoch in range(epochs_stage2):
        model.train()
        optimizer.zero_grad()
        out = model(features, adj)
        loss = ce_loss(out[train_mask], labels[train_mask])
        lam = final_lambda * (epoch / max(1, epochs_stage2-1))
        sp_loss = model.sparsity_loss()
        total_loss = loss + lam * sp_loss
        total_loss.backward()
        optimizer.step()

        if (epoch+1) % 10 == 0 or epoch == epochs_stage2-1:
            model.eval()
            with torch.no_grad():
                out_val = model(features, adj)
                pred = out_val.argmax(dim=1)
                val_acc = (pred[val_mask] == labels[val_mask]).float().mean().item()
                gates = {}
                for layer in model.layers:
                    if layer.gate is not None:
                        gates[layer.layer_name] = torch.sigmoid(layer.gate).item()
                k_vals = model.get_learned_K()
            print(f"\n📊 Epoch {epoch+1}/{epochs_stage2} | λ={lam:.3f} | Val Acc={val_acc:.4f}")
            print(f"   Gates: {gates}")
            print(f"   Learned K: {k_vals}")

    return model

def evaluate_gcn(model, adj, features, labels, mask):
    model.eval()
    t1 = time.perf_counter()
    with torch.no_grad():
        out = model(features, adj)
    t2 = time.perf_counter()
    pred = out.argmax(dim=1)
    acc = (pred[mask] == labels[mask]).float().mean().item()
    y_true = labels[mask].cpu().numpy()
    y_pred = pred[mask].cpu().numpy()
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'inference_time_ms': (t2 - t1) * 1000
    }

# -------------------- Main Experiment --------------------
def run_gcn_experiment(data_path, dataset_name, config, ep_baseline=100, ep_stage2=120, ep_fine=200):
    print("\n" + "="*80)
    print(f"DATASET: {dataset_name}")
    print(f"Description: {config['description']}")
    print(f"PCA Components: {config['pca_components']}")
    print(f"Patch Size: {config['patch_size']}")
    print("="*80)

    X, y = load_dataset(data_path, dataset_name, config)
    X_pca = applyPCA(X, config['pca_components'])
    nodes, node_labels, coords = extract_nodes_with_coords(X_pca, y, windowSize=config['patch_size'])
    print(f"Nodes: {nodes.shape}, Labels: {node_labels.shape}")
    n_classes = len(np.unique(node_labels))

    adj = build_adjacency(coords, max_dist=1)
    print(f"Adjacency matrix shape: {adj.shape}, number of edges: {adj._nnz()}")

    indices = np.arange(len(node_labels))
    train_idx, test_idx, _, _ = train_test_split(
        indices, node_labels, test_size=0.3, stratify=node_labels, random_state=SEED)
    val_idx, test_idx, _, _ = train_test_split(
        test_idx, node_labels[test_idx], test_size=0.5, stratify=node_labels[test_idx], random_state=SEED)

    train_mask = torch.zeros(len(node_labels), dtype=torch.bool)
    val_mask = torch.zeros(len(node_labels), dtype=torch.bool)
    test_mask = torch.zeros(len(node_labels), dtype=torch.bool)
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    features = torch.tensor(nodes, dtype=torch.float32)
    labels_t = torch.tensor(node_labels, dtype=torch.long)

    # ---- Baseline (no compression) ----
    print("\n--- Training Baseline GCN (No Compression) ---")
    baseline = SelfAdaptiveGCN(n_classes, nodes.shape[1],
                               hidden_dims=[256, 128],
                               compress_layers=[]).to(DEVICE)
    baseline_start = time.time()
    baseline = train_gcn_two_stage(baseline, adj, features, labels_t, train_mask, val_mask,
                                   epochs_stage1=ep_baseline, epochs_stage2=0, final_lambda=0.0, lr=0.005)
    baseline_train_time = time.time() - baseline_start
    baseline_metrics = evaluate_gcn(baseline, adj, features, labels_t, test_mask)
    baseline_params = sum(p.numel() for p in baseline.parameters())
    baseline_size_mb = baseline_params * 4 / (1024 * 1024)

    # ---- Self‑adaptive compressed GCN (compress only the first layer) ----
    print("\n--- Training Self-Adaptive Compressed GCN ---")
    compress_candidates = ['gcn1']   # only compress the first layer
    student = SelfAdaptiveGCN(n_classes, nodes.shape[1],
                              hidden_dims=[256, 128],
                              compress_layers=compress_candidates, gate_init=0.0).to(DEVICE)
    student_start = time.time()
    student = train_gcn_two_stage(student, adj, features, labels_t, train_mask, val_mask,
                                  epochs_stage1=120, epochs_stage2=ep_stage2, final_lambda=2.0, lr=0.005)
    student_train_time = time.time() - student_start

    final_learned_K = student.get_learned_K()
    print("\n--- Learned K values ---")
    for k, v in final_learned_K.items():
        print(f"  {k}: {v}")

    print("\n--- Finalizing Compression ---")
    student.finalize_compression(gate_threshold=0.4)

    # Fine-tune
    print("\n--- Fine-tuning Compressed GCN ---")
    opt_ft = optim.Adam(student.parameters(), lr=0.001, weight_decay=5e-4)
    for epoch in range(ep_fine):
        student.train()
        opt_ft.zero_grad()
        out = student(features, adj)
        loss = F.cross_entropy(out[train_mask], labels_t[train_mask])
        loss.backward()
        opt_ft.step()
        if (epoch+1) % 20 == 0:
            val_acc = evaluate_gcn(student, adj, features, labels_t, val_mask)['accuracy']
            print(f"  FT Epoch {epoch+1}: loss={loss.item():.4f}, val_acc={val_acc:.4f}")

    compressed_metrics = evaluate_gcn(student, adj, features, labels_t, test_mask)
    compressed_params = sum(p.numel() for p in student.parameters())
    compressed_size_mb = compressed_params * 4 / (1024 * 1024)
    reduction = (1 - compressed_params / baseline_params) * 100
    acc_change = (compressed_metrics['accuracy'] - baseline_metrics['accuracy']) * 100
    inf_change = (compressed_metrics['inference_time_ms'] - baseline_metrics['inference_time_ms']) / baseline_metrics['inference_time_ms'] * 100

    print("\n" + "="*80)
    print(f"FINAL RESULTS FOR {dataset_name} (GCN)")
    print("="*80)
    print(f"{'Metric':<20} {'Baseline':<18} {'Compressed':<18} {'Change':<12}")
    print("-"*70)
    print(f"{'Accuracy (%)':<20} {baseline_metrics['accuracy']*100:<18.2f} {compressed_metrics['accuracy']*100:<18.2f} {acc_change:>+11.2f}%")
    print(f"{'Precision (%)':<20} {baseline_metrics['precision']*100:<18.2f} {compressed_metrics['precision']*100:<18.2f}")
    print(f"{'Recall (%)':<20} {baseline_metrics['recall']*100:<18.2f} {compressed_metrics['recall']*100:<18.2f}")
    print(f"{'F1 Score (%)':<20} {baseline_metrics['f1']*100:<18.2f} {compressed_metrics['f1']*100:<18.2f}")
    print(f"{'Parameters':<20} {baseline_params:<18,} {compressed_params:<18,} {reduction:>+11.1f}%")
    print(f"{'Model Size (MB)':<20} {baseline_size_mb:<18.2f} {compressed_size_mb:<18.2f} {((1-compressed_size_mb/baseline_size_mb)*100):>+11.1f}%")
    print(f"{'Inference (ms)':<20} {baseline_metrics['inference_time_ms']:<18.2f} {compressed_metrics['inference_time_ms']:<18.2f} {inf_change:>+11.1f}%")
    print(f"{'Training Time (s)':<20} {baseline_train_time:<18.2f} {student_train_time:<18.2f} {((student_train_time/baseline_train_time-1)*100):>+11.1f}%")


Using device: cpu


In [2]:
config =  {
        'file': 'Botswana.mat',
        'gt_file': 'Botswana_gt.mat',
        'data_key': 'Botswana',
        'gt_key': 'Botswana_gt',
        'pca_components': 30,
        'patch_size': 5,
        'description': 'Botswana Delta, 14 classes'
    }
data_path = r"Data"
dataset_name = 'Botswana'
run_gcn_experiment(data_path, dataset_name, config, 100, 600, 200)


DATASET: Botswana
Description: Botswana Delta, 14 classes
PCA Components: 30
Patch Size: 5
Nodes: (3248, 30), Labels: (3248,)


/tmp/ipykernel_2648/2213838929.py:64: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  degree_inv_sqrt = np.power(degree, -0.5, where=degree>0)
/tmp/ipykernel_2648/2213838929.py:68: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  adj_indices = torch.tensor([adj_norm_coo.row, adj_norm_coo.col], dtype=torch.long)
/tmp/ipykernel_2648/2213838929.py:71: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytor

Adjacency matrix shape: torch.Size([3248, 3248]), number of edges: 22620

--- Training Baseline GCN (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.0542 | Val Acc: 0.9938
Stage1 Epoch 40/100 | Loss: 0.0091 | Val Acc: 1.0000
Stage1 Epoch 60/100 | Loss: 0.0063 | Val Acc: 0.9979
Early stopping at epoch 64 (best val_acc=1.0000)

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed GCN ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.1126 | Val Acc: 0.9897
Stage1 Epoch 40/120 | Loss: 0.0129 | Val Acc: 1.0000
Stage1 Epoch 60/120 | Loss: 0.0083 | Val Acc: 0.9979
Early stopping at epoch 67 (best val_acc=1.0000)

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 10/600 | λ=0.030 | Val Acc=1.0000
   Gates: {'gcn1': 0.4693123996257782}
   Learned K: {'gcn1': 156}

📊 Epoch 20/600 | λ=0.063 | Val Acc=0.9959
   Gates: {'gcn1': 0.4685865342617035}
   Learn

In [2]:
config =  {
        'file': 'Indian_pines_corrected.mat',
        'gt_file': 'Indian_pines_gt.mat',
        'data_key': 'indian_pines_corrected',
        'gt_key': 'indian_pines_gt',
        'pca_components': 30,
        'patch_size': 5,
        'description': 'Indian Pines, 16 classes'
    }
data_path = r"Data"
dataset_name = 'IndianPines'
run_gcn_experiment(data_path, dataset_name, config,  100, 400, 400)


DATASET: IndianPines
Description: Indian Pines, 16 classes
PCA Components: 30
Patch Size: 5
Nodes: (10086, 30), Labels: (10086,)


/tmp/ipykernel_3031/2213838929.py:64: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  degree_inv_sqrt = np.power(degree, -0.5, where=degree>0)
/tmp/ipykernel_3031/2213838929.py:68: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  adj_indices = torch.tensor([adj_norm_coo.row, adj_norm_coo.col], dtype=torch.long)
/tmp/ipykernel_3031/2213838929.py:71: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytor

Adjacency matrix shape: torch.Size([10086, 10086]), number of edges: 82732

--- Training Baseline GCN (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.7183 | Val Acc: 0.7832
Stage1 Epoch 40/100 | Loss: 0.3551 | Val Acc: 0.9055
Stage1 Epoch 60/100 | Loss: 0.2097 | Val Acc: 0.9438
Stage1 Epoch 80/100 | Loss: 0.1391 | Val Acc: 0.9689
Stage1 Epoch 100/100 | Loss: 0.1077 | Val Acc: 0.9795

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed GCN ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.9450 | Val Acc: 0.7191
Stage1 Epoch 40/120 | Loss: 0.4616 | Val Acc: 0.8724
Stage1 Epoch 60/120 | Loss: 0.2811 | Val Acc: 0.9240
Stage1 Epoch 80/120 | Loss: 0.1934 | Val Acc: 0.9570
Stage1 Epoch 100/120 | Loss: 0.1394 | Val Acc: 0.9722
Stage1 Epoch 120/120 | Loss: 0.1147 | Val Acc: 0.9795

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 10/400 | λ=0.045 | 

In [3]:
config =  {
        'file': 'GM01.mat',
        'gt_file': 'GM01.mat',
        'data_key': 'img',
        'gt_key': 'map',
        'pca_components': 15,
        'patch_size': 3,
        'description': 'Oil Spill Detection, 2 classes'
    }
data_path = r"Data"
dataset_name = 'GM01'
run_gcn_experiment(data_path, dataset_name, config, 100, 400, 20)


DATASET: GM01
Description: Oil Spill Detection, 2 classes
PCA Components: 15
Patch Size: 3
Nodes: (29282, 15), Labels: (29282,)


/tmp/ipykernel_3362/2213838929.py:64: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  degree_inv_sqrt = np.power(degree, -0.5, where=degree>0)
/tmp/ipykernel_3362/2213838929.py:68: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  adj_indices = torch.tensor([adj_norm_coo.row, adj_norm_coo.col], dtype=torch.long)
/tmp/ipykernel_3362/2213838929.py:71: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytor

Adjacency matrix shape: torch.Size([29282, 29282]), number of edges: 216632

--- Training Baseline GCN (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.0000 | Val Acc: 1.0000
Early stopping at epoch 31 (best val_acc=1.0000)

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed GCN ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.0000 | Val Acc: 1.0000
Early stopping at epoch 31 (best val_acc=1.0000)

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 10/400 | λ=0.045 | Val Acc=1.0000
   Gates: {'gcn1': 0.511694610118866}
   Learned K: {'gcn1': 156}

📊 Epoch 20/400 | λ=0.095 | Val Acc=1.0000
   Gates: {'gcn1': 0.5304371118545532}
   Learned K: {'gcn1': 153}

📊 Epoch 30/400 | λ=0.145 | Val Acc=1.0000
   Gates: {'gcn1': 0.5511366128921509}
   Learned K: {'gcn1': 149}

📊 Epoch 40/400 | λ=0.195 | Val Acc=1.0000
   Gates: {'gcn1': 0.5724486112594604

In [2]:
config =  {
        'file': 'Salinas_corrected.mat',
        'gt_file': 'Salinas_gt.mat',
        'data_key': 'salinas_corrected',
        'gt_key': 'salinas_gt',
        'pca_components': 24,
        'patch_size': 5,
        'description': 'Salinas Valley, 16 classes'
    }
data_path = r"Data"
dataset_name = 'Salinas'
run_gcn_experiment(data_path, dataset_name, config, 100, 400, 400)


DATASET: Salinas
Description: Salinas Valley, 16 classes
PCA Components: 24
Patch Size: 5
Nodes: (53370, 24), Labels: (53370,)


/tmp/ipykernel_3815/2213838929.py:64: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  degree_inv_sqrt = np.power(degree, -0.5, where=degree>0)
/tmp/ipykernel_3815/2213838929.py:68: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  adj_indices = torch.tensor([adj_norm_coo.row, adj_norm_coo.col], dtype=torch.long)
/tmp/ipykernel_3815/2213838929.py:71: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytor

Adjacency matrix shape: torch.Size([53370, 53370]), number of edges: 464484

--- Training Baseline GCN (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.2274 | Val Acc: 0.9319
Stage1 Epoch 40/100 | Loss: 0.0981 | Val Acc: 0.9686
Stage1 Epoch 60/100 | Loss: 0.0660 | Val Acc: 0.9819
Stage1 Epoch 80/100 | Loss: 0.0521 | Val Acc: 0.9869
Stage1 Epoch 100/100 | Loss: 0.0444 | Val Acc: 0.9893

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed GCN ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.3437 | Val Acc: 0.9146
Stage1 Epoch 40/120 | Loss: 0.1249 | Val Acc: 0.9614
Stage1 Epoch 60/120 | Loss: 0.0786 | Val Acc: 0.9786
Stage1 Epoch 80/120 | Loss: 0.0599 | Val Acc: 0.9848
Stage1 Epoch 100/120 | Loss: 0.0507 | Val Acc: 0.9879
Stage1 Epoch 120/120 | Loss: 0.0455 | Val Acc: 0.9891

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 10/400 | λ=0.045 |

In [2]:
config =  {
        'file': 'PaviaU.mat',
        'gt_file': 'PaviaU_gt.mat',
        'data_key': 'paviaU',
        'gt_key': 'paviaU_gt',
        'pca_components': 8,
        'patch_size': 5,
        'description': 'Pavia University, 9 classes'
    }
data_path = r"Data"
dataset_name = 'PaviaU'
run_gcn_experiment(data_path, dataset_name, config, 100, 400, 400)


DATASET: PaviaU
Description: Pavia University, 9 classes
PCA Components: 8
Patch Size: 5
Nodes: (41849, 8), Labels: (41849,)


/tmp/ipykernel_281/2213838929.py:64: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  degree_inv_sqrt = np.power(degree, -0.5, where=degree>0)
/tmp/ipykernel_281/2213838929.py:68: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  adj_indices = torch.tensor([adj_norm_coo.row, adj_norm_coo.col], dtype=torch.long)
/tmp/ipykernel_281/2213838929.py:71: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/

Adjacency matrix shape: torch.Size([41849, 41849]), number of edges: 334363

--- Training Baseline GCN (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.3839 | Val Acc: 0.8886
Stage1 Epoch 40/100 | Loss: 0.2334 | Val Acc: 0.9296
Stage1 Epoch 60/100 | Loss: 0.1859 | Val Acc: 0.9460
Stage1 Epoch 80/100 | Loss: 0.1584 | Val Acc: 0.9544
Stage1 Epoch 100/100 | Loss: 0.1441 | Val Acc: 0.9589

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed GCN ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.5273 | Val Acc: 0.8506
Stage1 Epoch 40/120 | Loss: 0.2904 | Val Acc: 0.9124
Stage1 Epoch 60/120 | Loss: 0.2098 | Val Acc: 0.9385
Stage1 Epoch 80/120 | Loss: 0.1765 | Val Acc: 0.9505
Stage1 Epoch 100/120 | Loss: 0.1598 | Val Acc: 0.9560
Stage1 Epoch 120/120 | Loss: 0.1434 | Val Acc: 0.9606

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 10/400 | λ=0.045 |

In [3]:
config = {
    'file': 'Pavia.mat',
    'gt_file': 'Pavia_gt.mat',
    'data_key': 'pavia',
    'gt_key': 'pavia_gt',
    'pca_components': 15,
    'patch_size': 5,
    'description': 'Pavia Centre, 9 classes'
}
data_path = r"Data"
dataset_name = 'Pavia'
run_gcn_experiment(data_path, dataset_name, config, 100, 400, 400)


DATASET: Pavia
Description: Pavia Centre, 9 classes
PCA Components: 15
Patch Size: 5
Nodes: (147275, 15), Labels: (147275,)


/tmp/ipykernel_281/2213838929.py:64: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  degree_inv_sqrt = np.power(degree, -0.5, where=degree>0)


Adjacency matrix shape: torch.Size([147275, 147275]), number of edges: 1211129

--- Training Baseline GCN (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.1012 | Val Acc: 0.9842
Stage1 Epoch 40/100 | Loss: 0.0348 | Val Acc: 0.9936
Stage1 Epoch 60/100 | Loss: 0.0245 | Val Acc: 0.9956
Stage1 Epoch 80/100 | Loss: 0.0209 | Val Acc: 0.9963
Stage1 Epoch 100/100 | Loss: 0.0195 | Val Acc: 0.9967

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed GCN ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.1936 | Val Acc: 0.9480
Stage1 Epoch 40/120 | Loss: 0.0522 | Val Acc: 0.9907
Stage1 Epoch 60/120 | Loss: 0.0310 | Val Acc: 0.9949
Stage1 Epoch 80/120 | Loss: 0.0256 | Val Acc: 0.9961
Stage1 Epoch 100/120 | Loss: 0.0231 | Val Acc: 0.9966
Stage1 Epoch 120/120 | Loss: 0.0212 | Val Acc: 0.9969

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 10/400 | λ=0.04